In [ ]:
# First generate all possibilities

import itertools as itr

C3_1_prim = [(1,0,0,0),(0,1,0,0),(0,0,1,0),(0,0,0,1)]
C3_2_prim = [(2,0,0,0),(0,2,0,0),(0,0,2,0),(0,0,0,2),\
             (1,1,0,0),(1,0,1,0),(1,0,0,1),(0,1,1,0),(0,1,0,1),(0,0,1,1)]

primaries = []
count_par = 0
count_dup = 0
for poss_prim in itr.product(C3_1_prim,C3_1_prim,C3_2_prim):
    
     # Check if selection rules are satisfied
    par = sum( j*poss_prim[i][j] for i,j in itr.product(range(3),range(1,4)) )
    if not par%2:
        count_par+=1
        continue

    # Now check to make sure this primary isn't already in the set of primaries with different labels
    Aposs_prim = tuple(tuple(poss_prim[i][j] for j in range(3,-1,-1)) for i in range(3))
    if Aposs_prim in primaries:
        count_dup+=1
        continue

    # If the primary has passed all checks, add it to the list
    primaries.append(poss_prim)

In [46]:
list(range(3,-1,-1))

[3, 2, 1, 0]

In [9]:
len(primaries)

40

In [24]:
# Now we wish to calculate the S-matrix

# To do this we need to first calculate the S-matrix of sp(6)_k
# To this end we recall the formula (K is some constant of proportionality):
# S_{\lambda,\mu} = K sum_{w\in W} sgn(w) exp(-\frac{2\pi i}{k+g^{\vee}} <w(\lambda+rho),\mu+\rho)>
# Note that, on the RHS, none of the objects are 'affine'

from sage.groups.matrix_gps.finitely_generated import MatrixGroup

# The rank of the algebra
rank = 3

# The dual Coxeter number
gvee = 4

# The quadratic form is
quadF = (1/2)*matrix(QQ,[[1,1,1],[1,2,2],[1,2,3]])

# The Weyl vector
rho = vector([1,1,1])

# The simple roots, coroots, and weights are as follows
simp_roots = [vector([2,-1,0]),vector([-1,2,-1]),vector([0,-2,2])]
simp_lengths = [(root*quadF*root)[0] for root in simp_roots]
simp_coroots = [2*simp_roots[i]/simp_lengths[i] for i in range(rank)]
simp_weights = [vector([1,0,0]),vector([0,1,0]),vector([0,0,1])]

# First let's find the Weyl group (not the affine weyl group)
# There are three simple Weyl reflections

# Calculate the matrix form of the simple Weyl reflections
simp_weyl = []
for i in range(rank):
    root = simp_roots[i]
    coroot = simp_coroots[i]
    # Find the columns of the matrix
    columns = [w - (coroot*quadF*w)[0]*root for w in simp_weights] 
    simp_weyl.append(matrix(columns).T)

# Use these to generate the full Weyl group
weyl_group = MatrixGroup(simp_weyl)
weyl_ref = [matrix(w) for w in list(weyl_group)]

# Now we are prepared to actually calculate the S-matrices of sp(6)_level
kac_C3_1 = [vector(l[1:]) for l in C3_1_prim]
kac_C3_2 = [vector(l[1:]) for l in C3_2_prim]
def S_matrix_sp6(level):
    K = I*8**(-1/2)*(level+gvee)**(-rank/2)
    if level==1:
        kac = kac_C3_1 
    elif level==2:
        kac = kac_C3_2
    S = matrix(SR,len(kac))
    for i,j in itr.product(range(len(kac)),repeat=2):
        S[i,j] = ( K*sum(w.det()*exp(-2*pi*I/(level+gvee)*((w*(kac[i]+rho))*quadF*(kac[j]+rho))[0]) for w in weyl_ref) ).simplify(algorithm='giac')
    
    return S

S1 = S_matrix_sp6(1)
S2 = S_matrix_sp6(2)

# Now we combine these into the S-matrix for the full theory
S_dsp6 = matrix(SR,len(primaries))
for i,j in itr.product(range(len(primaries)),repeat=2):
    pri = primaries[i]
    prj = primaries[j]
    S_dsp6[i,j] = ( 2*S1[C3_1_prim.index(pri[0]),C3_1_prim.index(prj[0])] * S1[C3_1_prim.index(pri[1]),C3_1_prim.index(prj[1])] * S2[C3_2_prim.index(pri[2]),C3_2_prim.index(prj[2])] ).simplify(algorithm='giac')

In [33]:
# Now we calculate the T-matrix and, by extension, the conformal weights
# First we calculate the modular anomalies and T-matrix of sp(6)_1 and sp(6)_2
moda1 = [ ((l+rho)*quadF*(l+rho))[0]/(2*(1+gvee)) - ((rho)*quadF*(rho))[0]/(2*gvee) for l in kac_C3_1 ]
moda2 = [ ((l+rho)*quadF*(l+rho))[0]/(2*(2+gvee)) - ((rho)*quadF*(rho))[0]/(2*gvee) for l in kac_C3_2 ]
T1 = diagonal_matrix( [ exp( 2*pi*I*m ) for m in moda1 ] )
ccT2 = diagonal_matrix( [ exp( -2*pi*I*m ) for m in moda2 ] )

# Now calculate the T matrix of the coset
T_dsp6 = matrix(SR,len(primaries))
for i,j in itr.product(range(len(primaries)),repeat=2):
    pri = primaries[i]
    prj = primaries[j]
    T_dsp6[i,j] = ( T1[C3_1_prim.index(pri[0]),C3_1_prim.index(prj[0])] * T1[C3_1_prim.index(pri[1]),C3_1_prim.index(prj[1])] * ccT2[C3_2_prim.index(pri[2]),C3_2_prim.index(prj[2])] )#.simplify(algorithm='giac')

In [44]:
moda2

[-7/24, 3/8, 7/8, 29/24, 0, 5/24, 1/3, 7/12, 17/24, 1]

In [43]:
ccT2

[ (1/4*I - 1/4)*sqrt(6) + (1/4*I + 1/4)*sqrt(2)                                              0                                              0                                              0                                              0                                              0                                              0                                              0                                              0                                              0]
[                                             0                         -(1/2*I + 1/2)*sqrt(2)                                              0                                              0                                              0                                              0                                              0                                              0                                              0                                              0]
[                                             0         

In [37]:
print( (T1*T1.H-identity_matrix(4)).n().norm() )
print( ((S1.n()*T1.n())**3 - S1.n()**2).n().norm() )
print( S1.is_symmetric() )
print( (S1*S1.H-identity_matrix(4)).n().norm() )

0.0
1.0819089225884229e-15
True
5.551115123125783e-17


In [40]:
print( (ccT2*ccT2.H-identity_matrix(10)).n().norm() )
print( ((S2.n()*ccT2.H.n())**3 - S2.n()**2).n().norm() )
print( S2.is_symmetric() )
print( (S2*S2.H-identity_matrix(10)).n().norm() )

0.0
5.470990389089858e-16
True
1.856963405217746e-16


In [39]:
print( (T_dsp6*T_dsp6.H-identity_matrix(40)).n().norm() )
print( ((S_dsp6.n()*T_dsp6.n())**3 - S_dsp6.n()**2).n().norm() )
print( S_dsp6.is_symmetric() )
print( (S_dsp6*S_dsp6.H-identity_matrix(40)).n().norm() )

2.220446049250313e-16
1.714775904795714
True
3.146158188370618e-16
